In [ ]:
!pip install openai anthropic pandas pillow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cp /content/drive/MyDrive/archive.zip /content/

In [ ]:
!mkdir -p /content/wikiart_tmp
!unzip -q /content/archive.zip -d /content/wikiart_tmp

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

#chiave API
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Incolla OPENAI_API_KEY e premi invio: ")

client = OpenAI()
print("OK: client inizializzato")

KeyboardInterrupt: 

In [ ]:
import os

ROOT_UNZIP = "/content/wikiart_tmp"
ext_ok = (".jpg", ".jpeg", ".png", ".webp")

best_dir = None
best_count = 0

for root, _, files in os.walk(ROOT_UNZIP):
    c = sum(1 for f in files if f.lower().endswith(ext_ok))
    if c > best_count:
        best_count = c
        best_dir = root

print("Cartella immagini trovata:", best_dir)
print("Numero immagini nella cartella:", best_count)

EXTRACT_DIR = best_dir

Cartella immagini trovata: /content/wikiart_tmp/wikiart/wikiart
Numero immagini nella cartella: 176436


In [ ]:
import pandas as pd
import os
from collections import defaultdict
import numpy as np

GEMINI_RESULTS = "results.csv"
df_g = pd.read_csv(GEMINI_RESULTS)
print("Gemini results:", df_g.shape)

# indicizza immagini
paths_by_name = defaultdict(list)

for root, _, files in os.walk(EXTRACT_DIR):
    for fn in files:
        if fn.lower().endswith(ext_ok):
            paths_by_name[fn].append(os.path.join(root, fn))

print("Nomi immagini indicizzati:", len(paths_by_name))

# match su filename
df_ready = df_g.copy()
images = df_ready["image"].astype(str).tolist()

image_paths = []
missing = 0
dups = 0

for img in images:
    cand = paths_by_name.get(img, [])
    if not cand:
        image_paths.append(np.nan)
        missing += 1
    else:
        if len(cand) > 1:
            dups += 1
        best = sorted(cand, key=lambda p: len(p))[0]
        image_paths.append(best)

df_ready["image_path"] = image_paths

print("Trovate:", df_ready["image_path"].notna().sum())
print("Mancanti:", missing)
print("Duplicati filename:", dups)

READY_PATH = "/content/openai_READY.csv"
df_ready_ok = df_ready[df_ready["image_path"].notna()].copy()
df_ready_ok.to_csv(READY_PATH, index=False)

print("Salvato:", READY_PATH, "righe:", len(df_ready_ok))
df_ready_ok[["image","image_path"]].head()

Gemini results: (7539, 27)
Nomi immagini indicizzati: 176436
Trovate: 7538
Mancanti: 1
Duplicati filename: 0
Salvato: /content/openai_READY.csv righe: 7538


,image,image_path
0,102839-frank-w-benson-river-scene-1921.jpg,/content/wikiart_tmp/wikiart/wikiart/102839-fr...
1,248823-badende-nach-tizian-1.jpg,/content/wikiart_tmp/wikiart/wikiart/248823-ba...
2,161158-rock-painting-motif.jpg,/content/wikiart_tmp/wikiart/wikiart/161158-ro...
3,39240-s-l1600.jpg,/content/wikiart_tmp/wikiart/wikiart/39240-s-l...
4,24258-francis-godolphin-2nd-earl-of-godolphin-...,/content/wikiart_tmp/wikiart/wikiart/24258-fra...


In [ ]:
PROMPT = r"""
You are an art history expert.

Analyze the provided artwork image carefully.

Respond ONLY in valid JSON format.
Do not include explanations outside the JSON.
If unsure, make your best reasoned hypothesis.
If a category does not fit any option, use "Unknown".

Provide:

1) artist_guess:
   - Provide exactly 3 possible artists.
   - Order them from most probable to least probable.

2) macro_style:
   Choose ONLY one from this list:
   [
   "Abstract (large family)",
   "Academic / Traditional",
   "Art Nouveau / Deco",
   "Asian Traditional",
   "Baroque",
   "Constructivist / Geometric Modern",
   "Contemporary Media",
   "Cubism",
   "Expressionism",
   "Feminist / Identity",
   "Futurism",
   "Impressionism",
   "Medieval",
   "Modernism",
   "Muralism",
   "Naturalism / Luminism / Tonalism",
   "Naïve",
   "Neoclassicism / Classicism",
   "Pop / Dada / Postmodern",
   "Realism",
   "Renaissance",
   "Romanticism",
   "Surrealism / Metaphysical",
   "Symbolism / Intimism",
   ]

3) technique:
   Choose ONLY one from:
   [
   "oil",
   "tempera",
   "fresco",
   "drawing",
   "pastel",
   "watercolour",
   "encaustic",
   ]

4) genre:
   Choose ONLY one:
   [
   "historical",
   "interior",
   "landscape",
   "mythological",
   "other",
   "genre"
   "portrait",
   "religious",
   "still-life",
   "study"
   ]

5) school:
   Choose ONLY one:
   [
   "american",
   "austrian",
   "belgian",
   "bohemian",
   "catalan",
   "danish",
   "dutch",
   "english",
   "finnish",
   "flemish",
   "french",
   "german",
   "greek",
   "hungarian",
   "irish",
   "italian",
   "netherlandish",
   "norwegian",
   "other",
   "polish",
   "portuguese",
   "russian",
   "scottish",
   "spanish",
   "swedish",
   "swiss"
   ]

6) historical_period:
   Choose one of these ranges if applicable.
   If clearly before or after, estimate the closest period.

   [
   "0751-0800","0801-0850","0851-0900","0951-1000",
   "1001-1050","1051-1100","1101-1150","1151-1200",
   "1201-1250","1251-1300","1301-1350","1351-1400",
   "1401-1450","1451-1500","1501-1550","1551-1600",
   "1601-1650","1651-1700","1701-1750","1751-1800",
   "1801-1850","1851-1900"
   ]

7) wolfflin_principles:

For each pair, rate the artwork on a scale from 1 to 5.

Use this scale for ALL pairs:

1 = Clearly first pole
2 = Mostly first pole
3 = Balanced / Borderline
4 = Mostly second pole
5 = Clearly second pole

Respond ONLY with an integer from 1 to 5 for each pair.

Pairs:

- linear_vs_painterly
  (1 = clearly linear, 5 = clearly painterly)

- plane_vs_recession
  (1 = clearly planar, 5 = clearly recessional)

- closed_vs_open_form
  (1 = clearly closed form, 5 = clearly open form)

- multiplicity_vs_unity
  (1 = clearly multiplicity, 5 = clearly unity)

- absolute_vs_relative_clarity
  (1 = clearly absolute clarity, 5 = clearly relative clarity)

8) short_description:
   Provide a brief (max 80 words) objective description of what is visible in the artwork.

Return strictly this JSON structure:

{
  "artist_guess": ["Name1", "Name2", "Name3"],
  "macro_style": "...",
  "technique": "...",
  "genre": "...",
  "school": "...",
  "historical_period": "...",
  "wolfflin_principles": {
      "linear_vs_painterly": "...",
      "plane_vs_recession": "...",
      "closed_vs_open_form": "...",
      "multiplicity_vs_unity": "...",
      "absolute_vs_relative_clarity": "..."
  },
  "short_description": "..."
}
"""
print("Prompt length:", len(PROMPT))

Prompt length: 3414


In [ ]:
import base64

def analyze_image_openai(image_path: str) -> str:
    #  codifica immagine
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")

    data_url = f"data:image/jpeg;base64,{img_b64}"

    resp = client.responses.create(
        model="gpt-4.1-mini",
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": PROMPT},
                {"type": "input_image", "image_url": data_url, "detail": "low"},
            ]
        }],
        max_output_tokens=700
    )

    return resp.output_text

In [ ]:
!mkdir -p /content/openai_run

In [ ]:
import os, json, time
import pandas as pd
from datetime import datetime

# CONFIG
MODEL_NAME = "gpt-4.1-mini"
SAVE_STATE_EVERY = 10
N_OPERE = 7545
MIN_SECONDS = 1.0

# INPUT
df_in = pd.read_csv(READY_PATH)

# OUTPUT
OUT_DIR = "/content/openai_run"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_JSONL = os.path.join(OUT_DIR, "results.jsonl")
STATE_JSON = os.path.join(OUT_DIR, "state.json")
OUT_CSV = os.path.join(OUT_DIR, "results.csv")

# resume
done_images = set()
if os.path.exists(OUT_JSONL):
    with open(OUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done_images.add(json.loads(line)["image"])
            except:
                pass

print("Gia fatti (da jsonl):", len(done_images))

processed_ok = 0
processed_err = 0

def save_state():
    st = {
        "done": len(done_images),
        "processed_ok": processed_ok,
        "processed_err": processed_err,
        "last_utc": datetime.utcnow().isoformat()
    }
    with open(STATE_JSON, "w", encoding="utf-8") as f:
        json.dump(st, f, ensure_ascii=False, indent=2)

with open(OUT_JSONL, "a", encoding="utf-8") as f_out:
    for _, row in df_in.head(N_OPERE).iterrows():
        image = str(row["image"])
        image_path = str(row["image_path"])

        if image in done_images:
            continue

        try:
            txt = analyze_image_openai(image_path)

            rec = {
                "model": MODEL_NAME,
                "timestamp_utc": datetime.utcnow().isoformat(),
                "image": image,
                "image_path": image_path,
                "openai_raw": txt
            }

            f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            done_images.add(image)
            processed_ok += 1

            if (processed_ok + processed_err) % SAVE_STATE_EVERY == 0:
                save_state()
                print("salvato state | ok:", processed_ok, "err:", processed_err, "done:", len(done_images))

            time.sleep(MIN_SECONDS)

        except Exception as e:
            processed_err += 1
            print("ERRORE su", image, "->", repr(e))
            time.sleep(max(3, MIN_SECONDS))

save_state()
print("FINE RUN | ok:", processed_ok, "err:", processed_err)
print("jsonl:", OUT_JSONL)
print("state:", STATE_JSON)

Gia fatti (da jsonl): 7537
ERRORE su 35889-screenshot-2021-12-21-152829.png -> NameError("name 'client' is not defined")
FINE RUN | ok: 0 err: 1
jsonl: /content/openai_run/results.jsonl
state: /content/openai_run/state.json


/tmp/ipykernel_8251/4005104163.py:42: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_utc": datetime.utcnow().isoformat()


In [ ]:
import json
import pandas as pd

rows = []
with open(OUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df_out = pd.DataFrame(rows)
df_out.to_csv(OUT_CSV, index=False)
print("Salvato:", OUT_CSV, "| righe:", len(df_out))
df_out.head()

Salvato: /content/openai_run/results.csv | righe: 7537


,model,timestamp_utc,image,image_path,openai_raw
0,gpt-4.1-mini,2026-03-04T13:45:55.044115,102839-frank-w-benson-river-scene-1921.jpg,/content/wikiart_tmp/wikiart/wikiart/102839-fr...,"{\n ""artist_guess"": [""Childe Hassam"", ""John H..."
1,gpt-4.1-mini,2026-03-04T13:46:01.779451,248823-badende-nach-tizian-1.jpg,/content/wikiart_tmp/wikiart/wikiart/248823-ba...,"```json\n{\n ""artist_guess"": [\n ""Jamini R..."
2,gpt-4.1-mini,2026-03-04T13:46:09.312242,161158-rock-painting-motif.jpg,/content/wikiart_tmp/wikiart/wikiart/161158-ro...,"{\n ""artist_guess"": [""Paul Klee"", ""Joan Miró""..."
3,gpt-4.1-mini,2026-03-04T13:46:14.790586,39240-s-l1600.jpg,/content/wikiart_tmp/wikiart/wikiart/39240-s-l...,"```json\n{\n ""artist_guess"": [""Francesco Haye..."
4,gpt-4.1-mini,2026-03-04T13:46:19.579277,24258-francis-godolphin-2nd-earl-of-godolphin-...,/content/wikiart_tmp/wikiart/wikiart/24258-fra...,"```json\n{\n ""artist_guess"": [""Hyacinthe Riga..."


In [ ]:
import json
import pandas as pd

rows = []

with open(OUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df_check = pd.DataFrame(rows)

print("Numero opere analizzate:", len(df_check))
df_check.head()

Numero opere analizzate: 7537


,model,timestamp_utc,image,image_path,openai_raw
0,gpt-4.1-mini,2026-03-04T13:45:55.044115,102839-frank-w-benson-river-scene-1921.jpg,/content/wikiart_tmp/wikiart/wikiart/102839-fr...,"{\n ""artist_guess"": [""Childe Hassam"", ""John H..."
1,gpt-4.1-mini,2026-03-04T13:46:01.779451,248823-badende-nach-tizian-1.jpg,/content/wikiart_tmp/wikiart/wikiart/248823-ba...,"```json\n{\n ""artist_guess"": [\n ""Jamini R..."
2,gpt-4.1-mini,2026-03-04T13:46:09.312242,161158-rock-painting-motif.jpg,/content/wikiart_tmp/wikiart/wikiart/161158-ro...,"{\n ""artist_guess"": [""Paul Klee"", ""Joan Miró""..."
3,gpt-4.1-mini,2026-03-04T13:46:14.790586,39240-s-l1600.jpg,/content/wikiart_tmp/wikiart/wikiart/39240-s-l...,"```json\n{\n ""artist_guess"": [""Francesco Haye..."
4,gpt-4.1-mini,2026-03-04T13:46:19.579277,24258-francis-godolphin-2nd-earl-of-godolphin-...,/content/wikiart_tmp/wikiart/wikiart/24258-fra...,"```json\n{\n ""artist_guess"": [""Hyacinthe Riga..."


In [ ]:
import random

sample = df_check.sample(min(5, len(df_check)))

for _, row in sample.iterrows():
    print("IMAGE:", row["image"])
    print("RISPOSTA MODELLO:")
    print(row["openai_raw"])
    print("\n" + "-"*80 + "\n")

IMAGE: 156648-diego-modi-ehrenburg.jpg
RISPOSTA MODELLO:
```json
{
  "artist_guess": ["Diego Rivera", "Unknown", "Unknown"],
  "macro_style": "Academic / Traditional",
  "technique": "drawing",
  "genre": "portrait",
  "school": "other",
  "historical_period": "1851-1900",
  "wolfflin_principles": {
    "linear_vs_painterly": 1,
    "plane_vs_recession": 2,
    "closed_vs_open_form": 2,
    "multiplicity_vs_unity": 3,
    "absolute_vs_relative_clarity": 1
  },
  "short_description": "A black and white line drawing depicting three men seated in a room with simple furniture and a table with teapots and dishes. There is a painting of a guitar on the wall behind them. The men have relaxed postures and are engaged in different activities such as smoking and holding their head."
}
```

--------------------------------------------------------------------------------

IMAGE: 212765-untitled-yellow-1969.jpg
RISPOSTA MODELLO:
```json
{
  "artist_guess": ["Ellsworth Kelly", "Josef Albers", "Agnes

PARTE CLAUDE

In [ ]:
import os
from getpass import getpass
from anthropic import Anthropic


if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Incolla ANTHROPIC_API_KEY e premi invio: ")

claude_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
print("OK: client Claude creato")

OK: client Claude creato


In [ ]:
import base64
import mimetypes

def analyze_image_claude(image_path: str, model_name: str = "claude-3-5-sonnet-latest") -> str:
    with open(image_path, "rb") as f:
        img_bytes = f.read()

    media_type, _ = mimetypes.guess_type(image_path)
    if media_type not in {"image/jpeg", "image/png", "image/webp", "image/gif"}:
        media_type = "image/jpeg"

    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    resp = claude_client.messages.create(
        model=model_name,
        max_tokens=700,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": img_b64,
                        },
                    },
                    {
                        "type": "text",
                        "text": PROMPT,
                    },
                ],
            }
        ],
    )

    parts = []
    for block in resp.content:
        if getattr(block, "type", None) == "text":
            parts.append(block.text)

    return "".join(parts)


In [ ]:
!mkdir -p /content/claude_run

In [ ]:
# Numero totale opere nel dataset pronto
tot_opere = len(df_ready)

# Opere gia analizzate
gia_fatte = len(done_images)

# Opere rimanenti da analizzare
restanti = tot_opere - gia_fatte

print("Totale opere nel dataset:", tot_opere)
print("Opere già analizzate:", gia_fatte)
print("Opere ancora da analizzare:", restanti)

Totale opere nel dataset: 7539
Opere già analizzate: 7537
Opere ancora da analizzare: 2


In [ ]:
import os, json, time
import pandas as pd
from datetime import datetime

# CONFIG
MODEL_NAME = "claude-haiku-4-5"
SAVE_STATE_EVERY = 10
N_OPERE = 5501
MIN_SECONDS = 1.0

# INPUT
df_in = pd.read_csv(READY_PATH)

# OUTPUT
OUT_DIR = "/content/claude_run"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_JSONL = os.path.join(OUT_DIR, "results_WikiArt_claude.jsonl")
STATE_JSON = os.path.join(OUT_DIR, "state_WikiArt_claude.json")
OUT_CSV = os.path.join(OUT_DIR, "results_WikiArt_calude.csv")

# resume
done_images = set()
if os.path.exists(OUT_JSONL):
    with open(OUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done_images.add(json.loads(line)["image"])
            except:
                pass

print("Gia fatti (da jsonl):", len(done_images))

processed_ok = 0
processed_err = 0

def save_state():
    st = {
        "done": len(done_images),
        "processed_ok": processed_ok,
        "processed_err": processed_err,
        "last_utc": datetime.utcnow().isoformat()
    }
    with open(STATE_JSON, "w", encoding="utf-8") as f:
        json.dump(st, f, ensure_ascii=False, indent=2)

with open(OUT_JSONL, "a", encoding="utf-8") as f_out:
    for _, row in df_in.head(N_OPERE).iterrows():
        image = str(row["image"])
        image_path = str(row["image_path"])

        if image in done_images:
            continue

        try:
            txt = analyze_image_claude(image_path, model_name=MODEL_NAME)

            rec = {
                "model": MODEL_NAME,
                "timestamp_utc": datetime.utcnow().isoformat(),
                "image": image,
                "image_path": image_path,
                "claude_raw": txt
            }

            f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            done_images.add(image)
            processed_ok += 1

            if (processed_ok + processed_err) % SAVE_STATE_EVERY == 0:
                save_state()
                print("salvato state | ok:", processed_ok, "err:", processed_err, "done:", len(done_images))

            time.sleep(MIN_SECONDS)

        except Exception as e:
            processed_err += 1
            print("ERRORE su", image, "->", repr(e))
            time.sleep(max(3, MIN_SECONDS))

save_state()
print("FINE RUN | ok:", processed_ok, "err:", processed_err)
print("jsonl:", OUT_JSONL)
print("state:", STATE_JSON)

Gia fatti (da jsonl): 5499
ERRORE su 143737-portrait-of-mucha-by-itself-1897.jpg -> BadRequestError("Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0.content.0.image.source.base64: The image was specified using the image/jpeg media type, but the image appears to be a image/png image'}, 'request_id': 'req_011CYsvaS12767rhQWWBEZ1o'}")


/tmp/ipykernel_8251/1082104476.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat(),


FINE RUN | ok: 1 err: 1
jsonl: /content/claude_run/results_WikiArt_claude.jsonl
state: /content/claude_run/state_WikiArt_claude.json


/tmp/ipykernel_8251/1082104476.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_utc": datetime.utcnow().isoformat()


In [ ]:
import json
import pandas as pd

rows = []
with open(OUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df_out = pd.DataFrame(rows)
df_out.to_csv(OUT_CSV, index=False)
print("Salvato:", OUT_CSV, "| righe:", len(df_out))
df_out.head()

Salvato: /content/claude_run/results_WikiArt_calude.csv | righe: 5500


,model,timestamp_utc,image,image_path,claude_raw
0,claude-sonnet-4-5,2026-03-07T13:26:00.727138,102839-frank-w-benson-river-scene-1921.jpg,/content/wikiart_tmp/wikiart/wikiart/102839-fr...,"```json\n{\n ""artist_guess"": [""John Henry Twa..."
1,claude-sonnet-4-5,2026-03-07T13:26:07.072260,248823-badende-nach-tizian-1.jpg,/content/wikiart_tmp/wikiart/wikiart/248823-ba...,"```json\n{\n ""artist_guess"": [""Jogen Chowdhur..."
2,claude-sonnet-4-5,2026-03-07T13:26:13.190646,161158-rock-painting-motif.jpg,/content/wikiart_tmp/wikiart/wikiart/161158-ro...,"```json\n{\n ""artist_guess"": [""Frank Stella"",..."
3,claude-sonnet-4-5,2026-03-07T13:26:19.268059,39240-s-l1600.jpg,/content/wikiart_tmp/wikiart/wikiart/39240-s-l...,"```json\n{\n ""artist_guess"": [""Alfred Chalon""..."
4,claude-sonnet-4-5,2026-03-07T13:26:25.367476,24258-francis-godolphin-2nd-earl-of-godolphin-...,/content/wikiart_tmp/wikiart/wikiart/24258-fra...,"```json\n{\n ""artist_guess"": [""Thomas Hudson""..."


In [ ]:
import json
import pandas as pd

rows = []

with open(OUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df_check = pd.DataFrame(rows)

print("Numero opere analizzate:", len(df_check))
df_check.head()


Numero opere analizzate: 918


,model,timestamp_utc,image,image_path,claude_raw
0,claude-sonnet-4-5,2026-03-07T13:26:00.727138,102839-frank-w-benson-river-scene-1921.jpg,/content/wikiart_tmp/wikiart/wikiart/102839-fr...,"```json\n{\n ""artist_guess"": [""John Henry Twa..."
1,claude-sonnet-4-5,2026-03-07T13:26:07.072260,248823-badende-nach-tizian-1.jpg,/content/wikiart_tmp/wikiart/wikiart/248823-ba...,"```json\n{\n ""artist_guess"": [""Jogen Chowdhur..."
2,claude-sonnet-4-5,2026-03-07T13:26:13.190646,161158-rock-painting-motif.jpg,/content/wikiart_tmp/wikiart/wikiart/161158-ro...,"```json\n{\n ""artist_guess"": [""Frank Stella"",..."
3,claude-sonnet-4-5,2026-03-07T13:26:19.268059,39240-s-l1600.jpg,/content/wikiart_tmp/wikiart/wikiart/39240-s-l...,"```json\n{\n ""artist_guess"": [""Alfred Chalon""..."
4,claude-sonnet-4-5,2026-03-07T13:26:25.367476,24258-francis-godolphin-2nd-earl-of-godolphin-...,/content/wikiart_tmp/wikiart/wikiart/24258-fra...,"```json\n{\n ""artist_guess"": [""Thomas Hudson""..."


In [ ]:
import random

sample = df_check.sample(min(5, len(df_check)))

for _, row in sample.iterrows():
    print("IMAGE:", row["image"])
    print("RISPOSTA MODELLO:")
    print(row["claude_raw"])
    print("\n" + "-"*80 + "\n")

IMAGE: 191675-girl-in-blue-with-turban-deborah-1940.jpg
RISPOSTA MODELLO:
```json
{
  "artist_guess": ["William H. Johnson", "Jacob Lawrence", "Romare Bearden"],
  "macro_style": "Expressionism",
  "technique": "oil",
  "genre": "portrait",
  "school": "american",
  "historical_period": "1851-1900",
  "wolfflin_principles": {
    "linear_vs_painterly": "2",
    "plane_vs_recession": "2",
    "closed_vs_open_form": "2",
    "multiplicity_vs_unity": "4",
    "absolute_vs_relative_clarity": "2"
  },
  "short_description": "A portrait of an African American woman wearing a vibrant blue jacket over a pink garment, with a colorful striped headband. Her hair is styled in a low bun. The figure is rendered in a simplified, expressive style with bold colors against a warm yellow-ochre background. The painting employs flat planes of color and strong outlines."
}
```

--------------------------------------------------------------------------------

IMAGE: 186809-ardoise-1981.jpg
RISPOSTA MODELLO:
